# Experiment 1: The Correlation Hierarchy

In [ ]:

import os
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import networkx as nx
from scipy.stats import pearsonr, spearmanr, kendalltau
import dcor
# from minepy import MINE
import matplotlib.pyplot as plt

from Helper_Functions import *


## Load the cleaned data for the experiment

In [ ]:

# Load saved Parquet files
# target_protein = None
# target_protein = "hba1c"
target_protein = "insulin"

wide_inxs=["person_id", "measurement_date", "visit_occurrence_id"]

if target_protein is None:
    file_cleaned = "data\\aou_labs_wide_cleaned.parquet"
else:
    # file_cleaned = f"data\\aou_{target_protein}_labs_wide_cleaned.parquet"
    file_cleaned = f"data\\aou_{target_protein}_labs_wide_cleaned_clustered.parquet"

print("Loading saved Parquet files...")
df_wide_cleaned = pd.read_parquet(file_cleaned, engine='pyarrow')

print(f"\n Loaded df_wide_cleaned: {df_wide_cleaned.shape[0]:,} rows x {df_wide_cleaned.shape[1]} columns")

df_wide_cleaned.head()


## Remove outliers by quantiles

In [ ]:
# cleaning low and high quantiles

df_wide_cleaned_quantile = df_wide_cleaned.copy()
q_l = 0.05
q_h = 0.95

for col in df_wide_cleaned_quantile.columns[len(wide_inxs):]:
    S = df_wide_cleaned_quantile[col].copy()
    df_wide_cleaned_quantile.loc[(S<S.quantile(q_l)) | (S>S.quantile(q_h)),col] = np.nan

df_wide_cleaned_quantile

In [ ]:

if target_protein == "hba1c":
    protein_name = "HbA1c"
if target_protein == "insulin":
    protein_name = "Insulin"


### see the outlier free columns perplexity

In [ ]:


cols_perplexity = get_columns_perplexity(df_wide_cleaned_quantile.iloc[:,len(wide_inxs):], n_bins = -1)

# cols_perplexity["Urobilinogen [Mass/Volume] In Urine"]
cols_perplexity.sort_values(ascending=True)


In [ ]:
# cols_perplexity.plot.kde(title="Perplexity Scores Density")
sns.histplot(cols_perplexity, log_scale=True, kde=True,)
cols_perplexity.describe()

### remove columns with low perplexity
Drop columns with a low number of effective values

In [ ]:
# remove columns with low perplexity

N = 10

# cols_perplexity.loc[cols_perplexity > N]
# cols_perplexity.loc[cols_perplexity > N].index
# cols_perplexity > N

df_wide_cleaned_quantile = df_wide_cleaned_quantile.loc[:,cols_perplexity.loc[cols_perplexity > N].index]
df_wide_cleaned_quantile

## See our main lab test cleaned distribution

In [ ]:

prot_data = df_wide_cleaned_quantile[protein_name].copy()
# prot_data = df_wide_cleaned[protein_name].copy()
# prot_data = prot_data[(prot_data>df_wide_cleaned[protein_name].quantile(0.05)) & (prot_data<df_wide_cleaned[protein_name].quantile(0.95))]
prot_data

sns.histplot(data=prot_data, log_scale=False, kde=True)
# sns.histplot(data=prot_data, log_scale=True, kde=True)

prot_data.describe()

# df_wide_cleaned[protein_name]

# sns.histplot(data=prot_data, x=protein_name, log_scale=False, kde=True)
# sns.histplot(data=df_wide_cleaned, x=protein_name, log_scale=True, kde=True)

## Calculate all tests' correlation (of all levels) between our main test and all other tests

In [ ]:
# Instantiate MINE object once outside the loop for efficiency
# mine = MINE()

min_needed_events = 30
min_individual_perplexity = 5
min_combined_perplexity = 30
binned_min_individual_perplexity = 5
binned_min_combined_perplexity = 30
bins = 20

df_connections = pd.DataFrame()

for col in df_wide_cleaned_quantile.columns[len(wide_inxs):]:
    if col != protein_name:
        # Isolate pairwise complete observations
        pair_df = df_wide_cleaned_quantile[[protein_name, col]].dropna()
        n_obs = len(pair_df)

        cols_perplexity = get_columns_perplexity(pair_df, n_bins = -1)
        binned_cols_perplexity = get_columns_perplexity(pair_df, n_bins = bins)
        
        x = pair_df[protein_name].values
        y = pair_df[col].values
        
        df_connections.loc[col, "num_events"] = n_obs
        df_connections.loc[col, "protein_perplexity"] = cols_perplexity[protein_name]
        df_connections.loc[col, "column_perplexity"] = cols_perplexity[col]
        df_connections.loc[col, "binned_protein_perplexity"] = binned_cols_perplexity[protein_name]
        df_connections.loc[col, "binned_column_perplexity"] = binned_cols_perplexity[col]

        perplexity_flag = False
        # check the two distributions have sufficient perplexity (good values distributions)
        if cols_perplexity[protein_name] > min_individual_perplexity and cols_perplexity[col] > min_individual_perplexity and (cols_perplexity[protein_name] * cols_perplexity[col]) > min_combined_perplexity:
            # check the two distributions have sufficient binned perplexity (good binning distributions)
            if binned_cols_perplexity[protein_name] > binned_min_individual_perplexity and binned_cols_perplexity[col] > binned_min_individual_perplexity and (binned_cols_perplexity[protein_name] * binned_cols_perplexity[col]) > binned_min_combined_perplexity:
                perplexity_flag = True
        
        # Valid execution check for variance and sample size
        if n_obs >= min_needed_events and perplexity_flag:
            # Pearson, Spearman, & Kendall Tau
            p_corr, p_pval = pearsonr(x, y)
            s_corr, s_pval = spearmanr(x, y)
            k_corr, k_pval = kendalltau(x, y)
            
            # Distance Correlation & t_test p-value
            d_corr = dcor.distance_correlation(x, y)
            # d_pval = dcor.independence.distance_covariance_test(x, y, num_resamples=500).pvalue
            d_pval = dcor.independence.distance_correlation_t_test(x, y).pvalue
            
            # # MIC & TIC Scores
            # mine.compute_score(x, y)
            # mic_score = mine.mic()
            # tic_score = mine.tic()

            # scores
            eps = 10**-3
            score_power = 4
            combined_perplexity = cols_perplexity[protein_name] * cols_perplexity[col]
            binned_combined_perplexity = binned_cols_perplexity[protein_name] * binned_cols_perplexity[col]
            
            linearly_score = np.log2(n_obs) * binned_combined_perplexity * np.abs(p_corr)**score_power / (p_pval + eps)
            monotonically_score = np.log2(n_obs) * binned_combined_perplexity * ((p_pval - s_pval) * np.abs(s_corr)**score_power / (s_pval + eps)) * ((p_pval - k_pval) * np.abs(k_corr)**score_power / (k_pval + eps))
            non_monotonically_score = np.log2(n_obs) * binned_combined_perplexity * (p_pval - d_pval) * (s_pval - d_pval) * (k_pval - d_pval) * np.abs(d_corr)**score_power / (d_pval + eps)
            non_score = (np.log2(n_obs) * binned_combined_perplexity) / ((np.abs(p_corr) + eps) * (np.abs(s_corr) + eps) * (np.abs(d_corr) + eps) * (np.abs(k_corr) + eps))
        else:
            p_corr = p_pval = s_corr = s_pval = k_corr = k_pval = np.nan
            d_corr = d_pval = mic_score = tic_score = np.nan
            linearly_score = monotonically_score = non_monotonically_score = 0
            non_score = 0

        # Assign metrics to dedicated columns
        df_connections.loc[col, "pearson_corr"] = p_corr
        df_connections.loc[col, "pearson_pvalue"] = p_pval
        df_connections.loc[col, "linearly_score"] = linearly_score
        df_connections.loc[col, "spearman_corr"] = s_corr
        df_connections.loc[col, "spearman_pvalue"] = s_pval
        df_connections.loc[col, "kendall_corr"] = k_corr
        df_connections.loc[col, "kendall_pvalue"] = k_pval
        df_connections.loc[col, "monotonically_score"] = monotonically_score
        df_connections.loc[col, "dcor"] = d_corr
        df_connections.loc[col, "dcor_pvalue"] = d_pval
        df_connections.loc[col, "non_monotonically_score"] = non_monotonically_score
        # df_connections.loc[col, "mic_score"] = mic_score
        # df_connections.loc[col, "tic_score"] = tic_score
        df_connections.loc[col, "non_score"] = non_score

df_connections

In [ ]:

# df_connections.sort_values(by=["pearson_pvalue"],ascending=True).head(50)
# df_connections.sort_values(by=["spearman_pvalue"],ascending=True).head(50)
# df_connections.sort_values(by=["kendall_pvalue"],ascending=True).head(50)
# df_connections.sort_values(by=["dcor_pvalue"],ascending=True).head(50)


### get adjusted p-value

In [ ]:

p = 0.05/len(df_connections)  # Bonferroni correction 
p
print(f"adjusted p-value : {p}")


### see linearly correlated lab tests

In [ ]:

print("linearly correlated")
print(len(df_connections.loc[df_connections["pearson_pvalue"] < p]))
# linearly_correlated = df_connections.loc[df_connections["pearson_pvalue"] < p].sort_values(by=["pearson_pvalue","num_events"],ascending=[True,False]).copy()
linearly_correlated = df_connections.loc[df_connections["pearson_pvalue"] < p].sort_values(by=["linearly_score"],ascending=False).copy()
linearly_correlated.head(20)


In [ ]:
plot_top_correlations(
    target_col=protein_name,
    stats_df=linearly_correlated,
    data_df=df_wide_cleaned_quantile,
    top_n=5,
    max_len=2000,
    fig_size=10.0,
    plot_type="kde", # can be either "kde" or "scatter"
    # save_name=f"results//{target_protein}_top_linearly correlated_columns",
    save_dir="results",
    file_format="png",
    dpi=300,
)

### see monotonically correlated lab tests

In [ ]:

print("monotonically correlated, but not linearly correlated")
print(len(df_connections.loc[(df_connections["pearson_pvalue"] > p) & ((df_connections["spearman_pvalue"] < p) | (df_connections["kendall_pvalue"] < p))]))
# monotonically_correlated = df_connections.loc[(df_connections["pearson_pvalue"] > p) & ((df_connections["spearman_pvalue"] < p) | (df_connections["kendall_pvalue"] < p))].sort_values(by=["spearman_pvalue","num_events"],ascending=[True,False]).copy()
monotonically_correlated = df_connections.loc[(df_connections["pearson_pvalue"] > p) & ((df_connections["spearman_pvalue"] < p) | (df_connections["kendall_pvalue"] < p))].sort_values(by=["monotonically_score"],ascending=False).copy()
monotonically_correlated.head(20)


In [ ]:
plot_top_correlations(
    target_col=protein_name,
    stats_df=monotonically_correlated,
    data_df=df_wide_cleaned_quantile,
    top_n=5,
    max_len=2000,
    fig_size=10.0,
    plot_type="kde", # can be either "kde" or "scatter"
    # save_dir=f"results//{target_protein}_top_monotonically_correlated_columns",
    save_dir="results",
    file_format="png",
    dpi=300,
)

### see non-monotonically correlated lab tests

In [ ]:

print("correlated but not monotonically")
print(len(df_connections.loc[((df_connections["pearson_pvalue"] > p) & (df_connections["spearman_pvalue"] > p) & (df_connections["kendall_pvalue"] > p)) & (df_connections["dcor_pvalue"] < p)]))
# non_monotonically_correlated = df_connections.loc[((df_connections["pearson_pvalue"] > p) & (df_connections["spearman_pvalue"] > p) & (df_connections["kendall_pvalue"] > p)) & (df_connections["dcor_pvalue"] < p)].sort_values(by=["dcor_pvalue","dcor"],ascending=[True,False]).copy()
non_monotonically_correlated = df_connections.loc[((df_connections["pearson_pvalue"] > p) & (df_connections["spearman_pvalue"] > p) & (df_connections["kendall_pvalue"] > p)) & (df_connections["dcor_pvalue"] < p)].sort_values(by=["non_monotonically_score"],ascending=False).copy()
non_monotonically_correlated.head(20)


In [ ]:
plot_top_correlations(
    target_col=protein_name,
    stats_df=non_monotonically_correlated,
    data_df=df_wide_cleaned_quantile,
    top_n=10,
    max_len=2000,
    fig_size=10.0,
    plot_type="kde", # can be either "kde" or "scatter"
    # save_dir=f"results//{target_protein}_top_non_monotonically_correlated_columns",
    save_dir="results",
    file_format="png",
    dpi=300,
)

### see non-correlated lab tests

In [ ]:

print("not correlated")
not_correlated = df_connections.loc[((df_connections["pearson_pvalue"] > p) & (df_connections["spearman_pvalue"] > p)) & (df_connections["dcor_pvalue"] > p)].sort_values(by=["non_score"],ascending=False).copy()
print(len(not_correlated))
not_correlated.head(20)




In [ ]:
plot_top_correlations(
    target_col=protein_name,
    stats_df=not_correlated,
    data_df=df_wide_cleaned_quantile,
    top_n=10,
    max_len=2000,
    fig_size=10.0,
    plot_type="kde", # can be either "kde" or "scatter"
    # save_dir=f"results//{target_protein}_top_not_correlated_columns",
    save_dir="results",
    file_format="png",
    dpi=300,
)

## Save final results

In [ ]:
with pd.ExcelWriter(f"results//{target_protein}_correlations.xlsx") as writer:
    df_connections.to_excel(writer, sheet_name="all_correlations", index_label="column_name", freeze_panes=(1, 6)) # freeze at F1
    linearly_correlated.to_excel(writer, sheet_name="linearly_correlated", index_label="column_name", freeze_panes=(1, 6))
    monotonically_correlated.to_excel(writer, sheet_name="monotonically_correlated", index_label="column_name", freeze_panes=(1, 6))
    non_monotonically_correlated.to_excel(writer, sheet_name="non_monotonically_correlated", index_label="column_name", freeze_panes=(1, 6))
    not_correlated.to_excel(writer, sheet_name="not_correlated", index_label="column_name", freeze_panes=(1, 6))